# C2Rust RL Training Notebook

**Fine-tunes DeepSeek-Coder-V2-Lite-Instruct (16B) with GRPO using the Rust compiler as the reward oracle.**

Target hardware: Nvidia L40S (48 GB VRAM) on HuggingFace Spaces or Colab A100.

---

## 1. Install dependencies

In [ ]:
# Install Rust toolchain
!curl --proto '=https' --tlsv1.2 -sSf https://sh.rustup.rs | sh -s -- -y --default-toolchain stable
import os
os.environ['PATH'] = os.environ['HOME'] + '/.cargo/bin:' + os.environ['PATH']
!rustc --version

In [ ]:
%%bash
pip install -q \
    unsloth \
    trl \
    datasets \
    wandb \
    pyyaml \
    tree-sitter \
    tree-sitter-c \
    matplotlib

## 2. Clone repo and verify

In [ ]:
!git clone https://github.com/YOUR_USERNAME/c2rust-rl.git
%cd c2rust-rl

In [ ]:
import sys
sys.path.insert(0, '.')

from tester.compiler import compile_and_evaluate

# Smoke test: compile a trivial Rust program
HELLO_RUST = 'fn main() { println!("Hello, Rust!"); }'
result = compile_and_evaluate(HELLO_RUST, reference_output='Hello, Rust!\n')
print(f'Success: {result.success}')
print(f'Reward:  {result.reward:.2f}')
assert result.success, 'rustc not working — check PATH'

## 3. Weights & Biases login

In [ ]:
import wandb
wandb.login()  # paste your API key when prompted

## 4. Preview training data

In [ ]:
from pathlib import Path

c_files = sorted(Path('data/c_programs').glob('*.c'))
print(f'{len(c_files)} C programs available for training:\n')
for f in c_files:
    lines = f.read_text().count('\n')
    print(f'  {f.name:30s}  ({lines} lines)')

## 5. Load model (DeepSeek-Coder-V2-Lite-Instruct, 4-bit QLoRA)

In [ ]:
from unsloth import FastLanguageModel

MODEL_NAME = 'deepseek-ai/DeepSeek-Coder-V2-Lite-Instruct'
MAX_SEQ_LEN = 4096

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LEN,
    dtype=None,
    load_in_4bit=True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj',
                    'gate_proj', 'up_proj', 'down_proj'],
    lora_alpha=32,
    lora_dropout=0.05,
    bias='none',
    use_gradient_checkpointing='unsloth',
    random_state=42,
)
print('Model loaded and LoRA adapters attached.')

## 6. Define reward function and dataset

In [ ]:
from tester.compiler import compile_and_evaluate
from train import build_grpo_dataset

def reward_fn(prompts, completions):
    """Compile each completion with rustc; return shaped rewards."""
    rewards = []
    for code in completions:
        r = compile_and_evaluate(code, timeout=30, run_clippy=True)
        rewards.append(r.reward)
    return rewards

dataset = build_grpo_dataset('data/c_programs')
print(f'Dataset size: {len(dataset)} examples')
print('First prompt (truncated):')
print(dataset[0]['prompt'][:400], '...')

## 7. GRPO Training

In [ ]:
from trl import GRPOConfig, GRPOTrainer

training_args = GRPOConfig(
    output_dir='checkpoints',
    num_train_epochs=50,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    learning_rate=1e-5,
    num_generations=4,           # GRPO group size
    max_completion_length=2048,
    temperature=0.2,
    logging_steps=10,
    save_steps=100,
    report_to='wandb',
)

trainer = GRPOTrainer(
    model=model,
    processing_class=tokenizer,
    reward_funcs=[reward_fn],
    args=training_args,
    train_dataset=dataset,
)

trainer.train()

## 8. Reward curve

In [ ]:
import matplotlib.pyplot as plt

# Pull reward history from trainer logs
history = []
for log in trainer.state.log_history:
    if 'train_loss' in log and 'reward' in log:
        history.append((log['step'], log['reward']))

if history:
    steps, rewards = zip(*history)
    plt.figure(figsize=(10, 4))
    plt.plot(steps, rewards, linewidth=2)
    plt.axhline(0, color='grey', linestyle='--', linewidth=0.8)
    plt.xlabel('Training step')
    plt.ylabel('Mean reward')
    plt.title('C2Rust GRPO — reward over training')
    plt.tight_layout()
    plt.savefig('reward_curve.png', dpi=150)
    plt.show()
else:
    print('No reward logs yet — run more training steps.')

## 9. Before vs After: untrained vs fine-tuned translation

In [ ]:
C_EXAMPLE = """
#include <stdio.h>
#include <stdlib.h>

int *create_array(int n) {
    int *arr = (int *)malloc(n * sizeof(int));
    for (int i = 0; i < n; i++) arr[i] = i * i;
    return arr;
}

int main(void) {
    int *a = create_array(5);
    for (int i = 0; i < 5; i++) printf("%d ", a[i]);
    printf("\\n");
    free(a);
    return 0;
}
"""

SYSTEM = (
    "You are an expert Rust programmer. Translate the following C code to safe, "
    "idiomatic Rust. Do NOT use unsafe blocks. Return only valid Rust source code."
)

prompt = f"{SYSTEM}\n\nC source:\n```c\n{C_EXAMPLE}\n```\n\nRust translation:\n```rust\n"

import torch
FastLanguageModel.for_inference(model)

inputs = tokenizer(prompt, return_tensors='pt').to(model.device)
with torch.no_grad():
    out = model.generate(**inputs, max_new_tokens=512, temperature=0.1, do_sample=True)

generated = tokenizer.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
print('=== Fine-tuned output ===')
print(generated)

In [ ]:
# Evaluate the generated code
from tester.compiler import compile_and_evaluate

result = compile_and_evaluate(generated, reference_output='0 1 4 9 16\n')
print(f'Compiled: {result.success}')
print(f'Reward:   {result.reward:.2f}')
print(f'Tests:    {result.test_passed}')
print(f'Unsafe:   {result.unsafe_count}')
if result.errors:
    print('Errors:')
    for e in result.errors:
        print(f'  [{e.code}] line {e.line}: {e.message}')

## 10. Save fine-tuned model

In [ ]:
trainer.save_model('final_model')
tokenizer.save_pretrained('final_model')
print('Model saved to ./final_model')

# Optional: push to HuggingFace Hub
# model.push_to_hub('YOUR_USERNAME/c2rust-deepseek-coder-v2-lite')
# tokenizer.push_to_hub('YOUR_USERNAME/c2rust-deepseek-coder-v2-lite')